In [4]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

df=pd.read_csv(r"C:\Users\NIKHIL\Downloads\data science assignment\data science assignment\adult_with_headers (1).csv")

df.head()

df.info()
df.describe()

df.replace(" ?", np.nan, inplace=True)
df.isnull().sum()

num_cols = df.select_dtypes(include=["int64","float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Standard Scaling
std_scaler = StandardScaler()
df_std = df.copy()
df_std[num_cols] = std_scaler.fit_transform(df[num_cols])

# Min-Max Scaling
mm_scaler = MinMaxScaler()
df_mm = df.copy()
df_mm[num_cols] = mm_scaler.fit_transform(df[num_cols])

# Identify low and high cardinality columns
low_card = [c for c in cat_cols if df[c].nunique() <= 5]
high_card = [c for c in cat_cols if df[c].nunique() > 5]

low_card, high_card

# One Hot Encoding
df_oh = pd.get_dummies(df, columns=low_card, drop_first=True)

# Label Encoding
df_le = df_oh.copy()

le = LabelEncoder()

for col in high_card:
    df_le[col] = le.fit_transform(df_le[col])

# ==========================================
# Feature Engineering
# ==========================================

# capital_intensity feature measures financial activity
# relative to working hours.

df_le["capital_intensity"] = (
    df_le["capital_gain"] + df_le["capital_loss"]
) / (df_le["hours_per_week"] + 1)

# high_hours feature identifies whether a person
# works more than 40 hours per week.

df_le["high_hours"] = np.where(df_le["hours_per_week"] > 40, 1, 0)

# ==========================================
# Log Transformation
# ==========================================

# capital_gain feature was highly positively skewed.
# Log transformation reduces skewness and minimizes
# the effect of extreme values.

df_le["log_capital_gain"] = np.log1p(df_le["capital_gain"])

df_le.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education_num   32561 non-null  int64 
 5   marital_status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital_gain    32561 non-null  int64 
 11  capital_loss    32561 non-null  int64 
 12  hours_per_week  32561 non-null  int64 
 13  native_country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,capital_gain,capital_loss,...,native_country,race_ Asian-Pac-Islander,race_ Black,race_ Other,race_ White,sex_ Male,income_ >50K,capital_intensity,high_hours,log_capital_gain
0,39,6,77516,9,13,4,0,1,2174,0,...,38,False,False,False,True,True,False,53.02439,0,7.684784
1,50,5,83311,9,13,2,3,0,0,0,...,38,False,False,False,True,True,False,0.00000,0,0.000000
2,38,3,215646,11,9,0,5,1,0,0,...,38,False,False,False,True,True,False,0.00000,0,0.000000
3,53,3,234721,1,7,2,5,0,0,0,...,38,False,True,False,False,True,False,0.00000,0,0.000000
4,28,3,338409,9,13,2,9,5,0,0,...,4,False,True,False,False,False,False,0.00000,0,0.000000


# Explanation of New Features

1. capital_intensity:
This feature was created to measure the relationship between capital gain, capital loss, and working hours. It helps capture the financial activity intensity of individuals relative to their work hours.

2. high_hours:
This binary feature identifies whether a person works more than 40 hours per week. It helps the model understand workload patterns and their possible relationship with income levels.

# Rationale for Log Transformation

The capital_gain feature was highly positively skewed, meaning most values were small while a few values were extremely large.

Log transformation was applied to reduce skewness, stabilize variance, and make the data distribution more normalized. This improves model performance and reduces the effect of outliers.